In [27]:
import sys
from pathlib import Path
# sys.path.insert(0, str(Path(__file__).parent.parent))

from typing import Optional

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import shap

import os
from sqlalchemy import (
    create_engine, Column, Integer, Float, String, Boolean,
    Date, DateTime, Text, ForeignKey, Index, UniqueConstraint, text,
inspect
)
from sqlalchemy.orm import DeclarativeBase, relationship, Session
from sqlalchemy.pool import StaticPool

pd.set_option('display.max_columns', None)

In [28]:
current_path = os.getcwd()
DB_PATH = str(Path(current_path).parent.parent / "dynasty_scout.db")
DB_PATH

'/Users/sandeeptiwari/Desktop/dynasty-ai-engine/dynasty_scout.db'

## Loading Data

In [29]:
def get_engine(db_path: Path = DB_PATH, echo: bool = False):
    """
    Returns a SQLAlchemy engine. Uses StaticPool so the same connection
    is reused in single-threaded contexts (fine for local use).
    """
    return create_engine(
        f"sqlite:///{db_path}",
        connect_args={"check_same_thread": False},
        poolclass=StaticPool,
        echo=echo,
    )

# LOAD NFL STATS
def load_players() -> pd.DataFrame:
    query = f"""
        SELECT * FROM players
    """
    with engine.connect() as conn:
        return pd.read_sql(text(query), conn)

def load_nfl_teams(seasons: list[int]) -> pd.DataFrame:
    query = f"""
        SELECT * FROM nfl_teams
        WHERE season IN ({','.join(map(str, seasons))})
    """
    with engine.connect() as conn:
        return pd.read_sql(text(query), conn)

def load_nfl_season_stats(seasons: list[int]) -> pd.DataFrame:
    query = f"""
        SELECT * FROM nfl_season_stats
        WHERE season IN ({','.join(map(str, seasons))})
    """
    with engine.connect() as conn:
        return pd.read_sql(text(query), conn)

def load_advanced_stats(seasons: list[int]) -> pd.DataFrame:
    query = f"""
        SELECT * FROM nfl_advanced_stats
        WHERE season IN ({','.join(map(str, seasons))})
    """
    with engine.connect() as conn:
        return pd.read_sql(text(query), conn)

def load_weekly_snaps(seasons: list[int]) -> pd.DataFrame:
    query = f"""
        SELECT * FROM nfl_weekly_snaps
        WHERE season IN ({','.join(map(str, seasons))})
    """
    with engine.connect() as conn:
        return pd.read_sql(text(query), conn)

def load_injuries(seasons: list[int]) -> pd.DataFrame:
    query = f"""
        SELECT * FROM injury_records
        WHERE season IN ({','.join(map(str, seasons))})
    """
    with engine.connect() as conn:
        return pd.read_sql(text(query), conn)

# LOAD COLLEGE STATS
def load_college_season_stats(seasons: list[int]) -> pd.DataFrame:
    query = f"""
        SELECT * FROM college_season_stats
        WHERE season IN ({','.join(map(str, seasons))})
    """
    with engine.connect() as conn:
        return pd.read_sql(text(query), conn)

def load_combine_measurements() -> pd.DataFrame:
    query = f"""
        SELECT * FROM combine_measurements
    """
    with engine.connect() as conn:
        return pd.read_sql(text(query), conn)

# LOAD SLEEPER DATA
def load_sleeper_league_snapshot() -> pd.DataFrame:
    query = f"""
        SELECT * FROM sleeper_league_snapshots
    """
    with engine.connect() as conn:
        return pd.read_sql(text(query), conn)

In [5]:
seasons = [2020]
engine = get_engine()

## NFL Data

In [ ]:
players_df = load_players()
nfl_teams_df = load_nfl_teams(seasons)
nfl_season_stats_df = load_nfl_season_stats(seasons)
advanced_stats_df = load_advanced_stats(seasons)
weekly_snaps_df = load_weekly_snaps(seasons)
injuries_df = load_injuries(seasons)

### Players Data

In [11]:
missing_pct = (
    players_df
    .isna()
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)
missing_pct.columns = ["feature", "missing_pct"]
print(missing_pct.to_string())

              feature  missing_pct
0         rookie_year     1.000000
1             cfbd_id     1.000000
2   depth_chart_order     1.000000
3          draft_year     1.000000
4          draft_team     1.000000
5            nfl_team     1.000000
6          draft_pick     1.000000
7          sleeper_id     1.000000
8         draft_round     0.512377
9       jersey_number     0.054167
10         birth_date     0.001593
11             weight     0.000613
12             height     0.000368
13          is_rookie     0.000000
14          player_id     0.000000
15          years_exp     0.000000
16             status     0.000000
17       college_team     0.000000
18           position     0.000000
19          last_name     0.000000
20         first_name     0.000000
21               name     0.000000
22            gsis_id     0.000000
23         updated_at     0.000000


### NFL Teams

In [13]:
nfl_teams_df

,team_abbr,season,full_name,head_coach,offensive_coordinator,defensive_coordinator,offensive_scheme,plays_per_game,pass_rate,pass_rate_neutral,team_pass_yards,team_rush_yards,team_total_tds,team_pass_attempts,team_targets,points_per_game,offensive_line_rank


In [12]:
missing_pct = (
    nfl_teams_df
    .isna()
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)
missing_pct.columns = ["feature", "missing_pct"]
print(missing_pct.to_string())

                  feature  missing_pct
0               team_abbr          NaN
1                  season          NaN
2               full_name          NaN
3              head_coach          NaN
4   offensive_coordinator          NaN
5   defensive_coordinator          NaN
6        offensive_scheme          NaN
7          plays_per_game          NaN
8               pass_rate          NaN
9       pass_rate_neutral          NaN
10        team_pass_yards          NaN
11        team_rush_yards          NaN
12         team_total_tds          NaN
13     team_pass_attempts          NaN
14           team_targets          NaN
15        points_per_game          NaN
16    offensive_line_rank          NaN


### NFL Season Stats

In [18]:
nfl_season_stats_df

,id,player_id,season,season_type,team,games,completions,attempts,passing_yards,passing_tds,interceptions,passing_epa,completion_pct,yards_per_attempt,passer_rating,sacks,sack_yards,carries,rushing_yards,rushing_tds,rushing_fumbles,rushing_epa,yards_per_carry,targets,receptions,receiving_yards,receiving_tds,receiving_fumbles,receiving_epa,yards_per_reception,catch_rate,yards_per_target,air_yards_total,yards_after_catch,fantasy_points_ppr,fantasy_points_half,fantasy_points_std,fantasy_ppg_ppr,fantasy_ppg_half,snap_pct,target_share,air_yards_share,racr,wopr,tgt_per_game
0,9,00-0019596,2020,REG,KC,16,401,610,4633,40,12,133.306174,0.657377,7.595082,102.172131,21,None,30,6,3,None,-18.186052,None,0,0,0,0,None,0.000000,NaN,NaN,NaN,0,0,337.92,None,None,21.120000,None,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,20,00-0020531,2020,REG,NO,12,275,390,2942,24,6,69.720447,0.705128,7.543590,106.378205,13,None,18,-2,2,None,-4.355731,None,0,0,0,0,None,0.000000,NaN,NaN,NaN,0,0,209.48,None,None,17.456667,None,0.898571,0.000000,0.000000,0.000000,0.000000,0.000000
2,45,00-0022127,2020,REG,LV,10,0,0,0,0,0,0.000000,NaN,NaN,NaN,0,None,0,0,0,None,0.000000,None,17,13,69,2,None,2.407178,5.307692,0.764706,4.058824,91,20,31.90,None,None,3.190000,None,0.375000,0.561207,0.384410,7.333333,1.110898,1.700000
3,54,00-0022787,2020,REG,ATL,1,0,0,0,0,0,0.000000,NaN,NaN,NaN,0,None,3,-4,0,None,0.000000,None,0,0,0,0,None,0.000000,NaN,NaN,NaN,0,0,-0.40,None,None,-0.400000,None,0.120000,0.000000,0.000000,0.000000,0.000000,0.000000
4,62,00-0022824,2020,REG,None,1,1,1,26,0,0,4.014011,1.000000,26.000000,193.750000,0,None,0,0,0,None,0.000000,None,0,0,0,0,None,0.000000,NaN,NaN,NaN,0,0,1.04,None,None,1.040000,None,NaN,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
629,5245,00-0036433,2020,REG,CAR,1,0,0,0,0,0,0.000000,NaN,NaN,NaN,0,None,4,24,0,None,1.731275,None,0,0,0,0,None,0.000000,NaN,NaN,NaN,0,0,2.40,None,None,2.400000,None,0.080000,0.000000,0.000000,0.000000,0.000000,0.000000
630,5250,00-0036439,2020,REG,DEN,5,0,0,0,0,0,0.000000,NaN,NaN,NaN,0,None,0,0,0,None,0.000000,None,9,6,63,0,None,2.915174,10.500000,0.666667,7.000000,79,18,12.30,None,None,2.460000,None,0.108000,0.241825,0.280559,3.200000,0.559129,1.800000
631,5253,00-0036442,2020,REG,CIN,10,264,404,2688,13,5,48.966725,0.653465,6.653465,89.830858,32,None,37,142,3,None,-0.141221,None,0,0,0,0,None,0.000000,NaN,NaN,NaN,0,0,173.72,None,None,17.372000,None,0.973000,0.000000,0.000000,0.000000,0.000000,0.000000
632,5258,00-0036450,2020,REG,TB,6,0,0,0,0,0,0.000000,NaN,NaN,NaN,0,None,26,109,0,None,-1.833543,None,10,5,34,1,None,-3.435261,6.800000,0.500000,3.400000,24,23,23.30,None,None,3.883333,None,0.140000,0.248911,0.068260,6.658730,0.421148,1.666667


In [14]:
missing_pct = (
    nfl_season_stats_df
    .isna()
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)
missing_pct.columns = ["feature", "missing_pct"]
print(missing_pct.to_string())

                feature  missing_pct
0       yards_per_carry     1.000000
1      fantasy_ppg_half     1.000000
2       rushing_fumbles     1.000000
3            sack_yards     1.000000
4   fantasy_points_half     1.000000
5    fantasy_points_std     1.000000
6     receiving_fumbles     1.000000
7        completion_pct     0.823344
8     yards_per_attempt     0.823344
9         passer_rating     0.823344
10  yards_per_reception     0.212934
11           catch_rate     0.171924
12     yards_per_target     0.171924
13             snap_pct     0.082019
14                 team     0.082019
15        receiving_tds     0.000000
16   fantasy_points_ppr     0.000000
17         target_share     0.000000
18    yards_after_catch     0.000000
19      air_yards_total     0.000000
20                 wopr     0.000000
21      air_yards_share     0.000000
22      fantasy_ppg_ppr     0.000000
23        receiving_epa     0.000000
24                 racr     0.000000
25                   id     0.000000
2

### NFL Advanced Stats

In [15]:
missing_pct = (
    advanced_stats_df
    .isna()
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)
missing_pct.columns = ["feature", "missing_pct"]
print(missing_pct.to_string())

                                 feature  missing_pct
0                max_completed_air_yards     1.000000
1            catch_pct_above_expectation     1.000000
2             avg_air_yards_differential     1.000000
3                 avg_intended_air_yards     0.820175
4       completion_pct_above_expectation     0.820175
5                         aggressiveness     0.820175
6                avg_air_yards_to_sticks     0.820175
7                avg_completed_air_yards     0.820175
8                      avg_time_to_throw     0.820175
9                             efficiency     0.758772
10  percent_attempts_gte_eight_defenders     0.758772
11                       avg_time_to_los     0.758772
12                        expected_yards     0.758772
13              rush_yards_over_expected     0.758772
14      rush_yards_over_expected_per_att     0.758772
15                           avg_cushion     0.421053
16            avg_intended_air_yards_rec     0.421053
17                        av

### Weekly Snaps

In [16]:
missing_pct = (
    weekly_snaps_df
    .isna()
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)
missing_pct.columns = ["feature", "missing_pct"]
print(missing_pct.to_string())

          feature  missing_pct
0              id          0.0
1       player_id          0.0
2          season          0.0
3            week          0.0
4         game_id          0.0
5            team          0.0
6   offense_snaps          0.0
7     offense_pct          0.0
8   defense_snaps          0.0
9     defense_pct          0.0
10       st_snaps          0.0


### Injuries

In [17]:
missing_pct = (
    injuries_df
    .isna()
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)
missing_pct.columns = ["feature", "missing_pct"]
print(missing_pct.to_string())

           feature  missing_pct
0   primary_injury     1.000000
1    report_status     0.547227
2               id     0.000000
3        player_id     0.000000
4           season     0.000000
5             week     0.000000
6             team     0.000000
7  practice_status     0.000000


## College Data

In [23]:
college_season_stats_df = load_college_season_stats(seasons)
combine_measurements_df = load_combine_measurements()

### College Season Stats

In [24]:
missing_pct = (
    college_season_stats_df
    .isna()
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)
missing_pct.columns = ["feature", "missing_pct"]
print(missing_pct.to_string())

                 feature  missing_pct
0           rush_carries     1.000000
1              player_id     1.000000
2           games_played     1.000000
3             pass_yards     1.000000
4       pass_completions     0.818429
5          pass_attempts     0.818429
6          interceptions     0.818429
7               pass_tds     0.818429
8               rush_tds     0.433126
9             receptions     0.253888
10       team_pass_yards     0.059876
11         team_pass_tds     0.059876
12        team_rec_yards     0.059876
13          team_rec_tds     0.059876
14               rec_tds     0.000000
15         yards_per_rec     0.000000
16             rec_yards     0.000000
17                    id     0.000000
18  rush_yards_per_carry     0.000000
19            rush_yards     0.000000
20              position     0.000000
21           player_name     0.000000
22            conference     0.000000
23                  team     0.000000
24                season     0.000000
25        cf

### Combine Measurements

In [25]:
missing_pct = (
    combine_measurements_df
    .isna()
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)
missing_pct.columns = ["feature", "missing_pct"]
print(missing_pct.to_string())

                    feature  missing_pct
0              catch_radius     1.000000
1                forty_yard     1.000000
2               speed_score     1.000000
3               sparq_score     1.000000
4                  position     1.000000
5                    school     1.000000
6                 player_id     1.000000
7               bench_press     0.505680
8                three_cone     0.405508
9                draft_year     0.389329
10                  shuttle     0.388296
11               broad_jump     0.215146
12            vertical_jump     0.198623
13                   height     0.003787
14                      bmi     0.003787
15                   weight     0.003442
16  relative_athletic_score     0.002065
17              player_name     0.000000
18                       id     0.000000


## Sleeper Data

In [30]:
sleeper_snapshot_df = load_sleeper_league_snapshot()

In [32]:
sleeper_snapshot_df

,id,snapshot_date,sleeper_player_id,player_name,position,dynasty_adp,redraft_adp,trending_adds,trending_drops,percent_owned,percent_started


In [31]:
missing_pct = (
    sleeper_snapshot_df
    .isna()
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)
missing_pct.columns = ["feature", "missing_pct"]
print(missing_pct.to_string())

              feature  missing_pct
0                  id          NaN
1       snapshot_date          NaN
2   sleeper_player_id          NaN
3         player_name          NaN
4            position          NaN
5         dynasty_adp          NaN
6         redraft_adp          NaN
7       trending_adds          NaN
8      trending_drops          NaN
9       percent_owned          NaN
10    percent_started          NaN
